# GuardianEye — Google Colab Orchestration Notebook

This notebook is a **thin orchestration wrapper only**.
All project logic lives in the repository's Python modules.

**Steps performed:**
1. Authenticate with GitHub (private repository)
2. Clone the repository
3. Install declared dependencies
4. Mount Google Drive and configure paths
5. Run `make` targets

> **Security note:** Never paste your GitHub token into a notebook cell.
> Store it as a Colab Secret (🔑 icon → Add secret → `GITHUB_TOKEN`).

## 0. Configuration — edit before running

Set these variables before executing any cells.

In [ ]:
# ---------------------------------------------------------------------------
# User-configurable settings — adjust before running
# ---------------------------------------------------------------------------
REPO_OWNER  = "your-github-username"   # e.g. "jdoe"
REPO_NAME   = "guardian-eye"
BRANCH      = "main"

# Google Drive path where datasets have been uploaded
# Leave empty to skip Drive mounting
GDRIVE_DATA_PATH = ""   # e.g. "/content/drive/MyDrive/guardian-eye/data"

# Where to write results / checkpoints on Drive
GDRIVE_RESULTS_PATH = ""  # e.g. "/content/drive/MyDrive/guardian-eye/results"

## 1. Authenticate with GitHub

Store your PAT in Colab Secrets (`🔑` icon) as `GITHUB_TOKEN`.
The cell below reads it without printing or persisting it.

In [ ]:
from google.colab import userdata

# Read the token from Colab Secrets — never printed or stored in the notebook
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN secret not found.\n"
        "Add it via the 🔑 Secrets panel before running."
    )
print("GitHub token loaded (not displayed).")

## 2. Clone the repository

In [ ]:
import os
import subprocess

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
CLONE_DIR = f"/content/{REPO_NAME}"

if not os.path.isdir(CLONE_DIR):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, CLONE_DIR],
        check=True,
    )
    print(f"Repository cloned to {CLONE_DIR}")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "origin", BRANCH], check=True)
    print(f"Repository updated at {CLONE_DIR}")

os.chdir(CLONE_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Install dependencies

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "-e", ".[dev]"], check=True)
print("Dependencies installed.")

## 4. Mount Google Drive and set paths

In [ ]:
import os

if GDRIVE_DATA_PATH:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_ROOT"] = GDRIVE_DATA_PATH
    print(f"DATA_ROOT  = {GDRIVE_DATA_PATH}")
else:
    print("Google Drive not configured — using default data/ directory.")

if GDRIVE_RESULTS_PATH:
    os.makedirs(GDRIVE_RESULTS_PATH, exist_ok=True)
    os.environ["RESULTS_ROOT"] = GDRIVE_RESULTS_PATH
    print(f"RESULTS_ROOT = {GDRIVE_RESULTS_PATH}")
else:
    print("RESULTS_ROOT not configured — using default results/ directory.")

## 5. Verify environment

In [ ]:
import subprocess
subprocess.run(["make", "data-check"], check=True)

## 6. Run the smoke test (fast CPU self-check)

In [ ]:
import subprocess
subprocess.run(["make", "smoke"], check=True)

## 7. Run tests and lint

In [ ]:
import subprocess
subprocess.run(["make", "test"], check=True)
subprocess.run(["make", "lint"], check=True)

## 8. Phase 1+ training targets

Uncomment the relevant target when the corresponding phase is ready.
All targets are defined in the `Makefile`.

In [ ]:
import subprocess

# Uncomment and run the appropriate target:
# subprocess.run(["make", "train-image"], check=True)
# subprocess.run(["make", "train-audio"], check=True)
# subprocess.run(["make", "pairs"], check=True)
# subprocess.run(["make", "fuse"], check=True)
# subprocess.run(["make", "evaluate"], check=True)
# subprocess.run(["make", "bda"], check=True)

print("Training targets are available in Phase 1+. See Makefile for details.")